# Credit Card Fraud Detection Using Machine Learning

**Objective**: Detect fraudulent credit card transactions accurately from imbalanced data using Scikit-Learn, Robust Scaling, Cost-Sensitive Ensemble Learning, and comprehensive evaluation metrics (PR-AUC, ROC-AUC, Recall, Precision).

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

## 2. Load and Automatically Inspect Dataset

We inspect:
- Total number of rows and columns
- Missing values
- Duplicate records
- Target column (`Class`) distribution

In [ ]:
data_path = os.path.join("..", "data", "creditcard.csv")
if not os.path.exists(data_path):
    # Fallback if running from repository root
    data_path = "data/creditcard.csv"

print(f"Loading data from: {data_path}")
df = pd.read_csv(data_path)

# Automatic Inspection
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Rows (Transactions) : {df.shape[0]:,}")
print(f"Columns (Features)  : {df.shape[1]}")
print(f"Missing Values      : {df.isnull().sum().sum()}")
print(f"Duplicate Rows      : {df.duplicated().sum():,}")

class_counts = df['Class'].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100
print(f"Legitimate (Class 0): {class_counts[0]:,} ({100 - fraud_pct:.3f}%)")
print(f"Fraudulent (Class 1): {class_counts[1]:,} ({fraud_pct:.3f}%)")
print("=" * 50)
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Visualizing the Extreme Class Imbalance
plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#2b5c8f", "#d9534f"])
plt.title("Class Distribution (0 = Legitimate, 1 = Fraudulent)", fontsize=12, fontweight="bold")
plt.yscale("log")
plt.ylabel("Count (Log Scale)")
plt.xlabel("Class")
plt.show()

In [ ]:
# Transaction Amount Distribution by Class
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df['Class'] == 0]['Amount'], bins=40, kde=True, ax=ax1, color="#2b5c8f")
ax1.set_title("Amount Distribution: Legitimate", fontweight="bold")
ax1.set_yscale("log")

sns.histplot(df[df['Class'] == 1]['Amount'], bins=40, kde=True, ax=ax2, color="#d9534f")
ax2.set_title("Amount Distribution: Fraudulent", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Feature Scaling

Because features `V1` to `V28` are already PCA-transformed (centered with unit variance), we only need to scale `Time` and `Amount`.
We use `RobustScaler` because transaction amounts have extreme positive outliers.

In [ ]:
# Remove duplicates
df_clean = df.drop_duplicates().copy()

X = df_clean.drop(columns=["Class"])
y = df_clean["Class"].astype(int)

# Stratified 80/20 train-test split preserving minority class proportion
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale 'Amount' and 'Time'
scaler = RobustScaler()
cols_to_scale = ["Amount", "Time"]
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print(f"Train set: {X_train.shape[0]:,} records ({y_train.sum()} frauds)")
print(f"Test set : {X_test.shape[0]:,} records ({y_test.sum()} frauds)")

## 5. Model Training with Class Weight Balancing

We train two cost-sensitive classifiers:
1. **Balanced Logistic Regression**
2. **Balanced Random Forest Classifier**

In [ ]:
# 1. Logistic Regression
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print("Models trained successfully!")

## 6. Model Evaluation & Comparison

In [ ]:
def evaluate_performance(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, y_pred, digits=4))
    print(f"ROC-AUC Score : {roc_auc_score(y_test, y_proba):.4f}")
    print(f"PR-AUC Score  : {average_precision_score(y_test, y_proba):.4f}\n")
    return y_pred, y_proba

lr_pred, lr_proba = evaluate_performance(lr, X_test, y_test, "Logistic Regression")
rf_pred, rf_proba = evaluate_performance(rf, X_test, y_test, "Random Forest")

### Confusion Matrix & Curves

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, rf_pred), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Random Forest Confusion Matrix", fontweight="bold")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, rf_proba)
ax1.plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc_score(y_test, rf_proba):.3f})")
ax1.plot([0, 1], [0, 1], 'k--')
ax1.set_title("ROC Curve", fontweight="bold")
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.legend()

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, rf_proba)
ax2.plot(rec, prec, color="#d9534f", label=f"PR Curve (AUC = {average_precision_score(y_test, rf_proba):.3f})")
ax2.set_title("Precision-Recall Curve", fontweight="bold")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importances = rf.feature_importances_
top_indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(9, 5))
sns.barplot(x=importances[top_indices], y=X.columns[top_indices], palette="viridis")
plt.title("Top 15 Most Discriminant Features for Fraud Detection", fontweight="bold")
plt.xlabel("Relative Importance")
plt.show()

## 8. Export Model Artifacts

In [ ]:
models_dir = os.path.join("..", "models")
os.makedirs(models_dir, exist_ok=True)

joblib.dump(rf, os.path.join(models_dir, "fraud_model.joblib"))
joblib.dump(scaler, os.path.join(models_dir, "scaler.joblib"))
print("Model and Scaler successfully exported to models/")